In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [2]:
DATA_PATH = Path("../data/StudentsPrepared.xlsx")
df = pd.read_excel(DATA_PATH)
df.shape
df.info()
df.isnull().sum()
df.isnull().sum()[df.isnull().sum() > 0]
df["Target"].value_counts()


<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 28 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   EstadoCivil                                 4424 non-null   str    
 1   Curso                                       4424 non-null   str    
 2   QualificacaoAnterior                        4424 non-null   str    
 3   QualificacaoAnteriorGrau                    4424 non-null   float64
 4   Nacionalidade                               4424 non-null   str    
 5   NotaAdmissao                                4424 non-null   float64
 6   NecessidadesEspeciais                       4424 non-null   int64  
 7   Devedor                                     4424 non-null   int64  
 8   MensalidadesEmDia                           4424 non-null   int64  
 9   Genero                                      4424 non-null   str    
 10  Bolsista               

Target
Graduado       2209
Desistente     1421
Matriculado     794

In [3]:
df["Evasao"] = (df["Target"] == "Desistente").astype(int)
df["Evasao"].value_counts()


Evasao
0    3003
1    1421

In [4]:
# Separar variável alvo e variáveis explicativas
X = df.drop(columns=["Target", "Evasao"])
y = df["Evasao"]

# Identificar colunas categóricas, booleanas/binárias e numéricas contínuas
colunas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()

colunas_booleanas = [
    "NecessidadesEspeciais",
    "Devedor",
    "MensalidadesEmDia",
    "Bolsista",
    "International",
]

colunas_numericas = [
    coluna for coluna in X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    if coluna not in colunas_booleanas
]

print("Colunas categóricas:")
print(colunas_categoricas)

print("\nColunas booleanas/binárias:")
print(colunas_booleanas)

print("\nColunas numéricas:")
print(colunas_numericas)

Colunas categóricas:
['EstadoCivil', 'Curso', 'QualificacaoAnterior', 'Nacionalidade', 'Genero']

Colunas booleanas/binárias:
['NecessidadesEspeciais', 'Devedor', 'MensalidadesEmDia', 'Bolsista', 'International']

Colunas numéricas:
['QualificacaoAnteriorGrau', 'NotaAdmissao', 'UnidadesCurriculares1SemestreCreditado', 'UnidadesCurriculares1SemestreInscrito', 'UnidadesCurriculares1SemestreAvaliacoes', 'UnidadesCurriculares1SemestreAprovado', 'UnidadesCurriculares1SemestreGrau', 'UnidadesCurriculares1SemestreSemAvaliacoes', 'UnidadesCurriculares2SemestreCreditado', 'UnidadesCurriculares2SemestreInscrito', 'UnidadesCurriculares2SemestreAvaliacoes', 'UnidadesCurriculares2SemestreAprovado', 'UnidadesCurriculares2SemestreGrau', 'UnidadesCurriculares2SemestreSemAvaliacoes', 'TaxaDesemprego', 'TaxaInflacao', 'PIB']


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas),
        ("num", StandardScaler(), colunas_numericas),
        ("bool", "passthrough", colunas_booleanas)
    ]
)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced"
))
])

In [8]:
pipeline_rf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['EstadoCivil', 'Curso',
                                                   'QualificacaoAnterior',
                                                   'Nacionalidade', 'Genero']),
                                                 ('num', StandardScaler(),
                                                  ['QualificacaoAnteriorGrau',
                                                   'NotaAdmissao',
                                                   'UnidadesCurriculares1SemestreCreditado',
                                                   'UnidadesCurriculares1SemestreInscrito',
                                                   'UnidadesCurricular...
                                                   'UnidadesCurriculares2SemestreGrau',
                      

In [9]:
y_pred = pipeline_rf.predict(X_test)
y_proba = pipeline_rf.predict_proba(X_test)[:, 1]

In [10]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

print("Acurácia:")
print(accuracy_score(y_test, y_pred))

print("\nROC AUC:")
print(roc_auc_score(y_test, y_proba))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

Acurácia:
0.8677966101694915

ROC AUC:
0.9238358604204262

Matriz de confusão:
[[531  70]
 [ 47 237]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.92      0.88      0.90       601
           1       0.77      0.83      0.80       284

    accuracy                           0.87       885
   macro avg       0.85      0.86      0.85       885
weighted avg       0.87      0.87      0.87       885



In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    pipeline_rf,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

In [12]:
for metric in scoring.keys():
    train_scores = cv_results[f"train_{metric}"]
    test_scores = cv_results[f"test_{metric}"]

    print(f"\n{metric.upper()}")
    print(f"Treino: média = {train_scores.mean():.4f} | desvio = {train_scores.std():.4f}")
    print(f"Validação: média = {test_scores.mean():.4f} | desvio = {test_scores.std():.4f}")


ACCURACY
Treino: média = 0.8677 | desvio = 0.0037
Validação: média = 0.8421 | desvio = 0.0152

PRECISION
Treino: média = 0.7701 | desvio = 0.0072
Validação: média = 0.7356 | desvio = 0.0266

RECALL
Treino: média = 0.8386 | desvio = 0.0032
Validação: média = 0.7951 | desvio = 0.0229

F1
Treino: média = 0.8029 | desvio = 0.0047
Validação: média = 0.7639 | desvio = 0.0212

ROC_AUC
Treino: média = 0.9378 | desvio = 0.0022
Validação: média = 0.9078 | desvio = 0.0110


## Tratamento paralelo das colunas de grau


In [13]:
colunas_grau = [
    "UnidadesCurriculares1SemestreGrau",
    "UnidadesCurriculares2SemestreGrau",
]

def corrigir_grau(valor):
    if pd.isna(valor):
        return valor

    valor = float(valor)

    while abs(valor) > 20:
        valor /= 10

    return valor

df_tratado = df.copy()

for coluna in colunas_grau:
    df_tratado[coluna] = df_tratado[coluna].apply(corrigir_grau)

resumo_tratamento = pd.DataFrame({
    "coluna": colunas_grau,
    "valores_corrigidos": [
        (df[coluna] != df_tratado[coluna]).sum()
        for coluna in colunas_grau
    ],
    "maximo_antes": [df[coluna].max() for coluna in colunas_grau],
    "maximo_depois": [df_tratado[coluna].max() for coluna in colunas_grau],
})

resumo_tratamento

                              coluna  valores_corrigidos  maximo_antes  maximo_depois
0  UnidadesCurriculares1SemestreGrau                1790  1.733333e+16      18.875000
1  UnidadesCurriculares2SemestreGrau                1675  1.857143e+16      18.571429

In [14]:
comparacao_graus = pd.concat(
    [
        df[colunas_grau].describe().T.add_suffix("_sem_tratamento"),
        df_tratado[colunas_grau].describe().T.add_suffix("_com_tratamento"),
    ],
    axis=1
)

comparacao_graus

                                   count_sem_tratamento  mean_sem_tratamento  std_sem_tratamento  min_sem_tratamento  25%_sem_tratamento  50%_sem_tratamento  75%_sem_tratamento  max_sem_tratamento  count_com_tratamento  mean_com_tratamento  std_com_tratamento  min_com_tratamento  25%_com_tratamento  50%_com_tratamento  75%_com_tratamento  max_com_tratamento
UnidadesCurriculares1SemestreGrau                4424.0         4.458091e+15        6.172473e+15                 0.0               11.25                13.5        1.216667e+16        1.733333e+16                4424.0            10.640822            4.843663                 0.0               11.00           12.285714           13.400000           18.875000
UnidadesCurriculares2SemestreGrau                4424.0         3.928634e+15        5.970124e+15                 0.0               11.00                13.0        1.166667e+16        1.857143e+16                4424.0            10.230206            5.210808                 0.0 

In [15]:
X_tratado = df_tratado.drop(columns=["Target", "Evasao"])
y_tratado = df_tratado["Evasao"]

X_train_tratado, X_test_tratado, y_train_tratado, y_test_tratado = train_test_split(
    X_tratado,
    y_tratado,
    test_size=0.2,
    random_state=42,
    stratify=y_tratado
)

In [16]:
pipeline_rf_tratado = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced"
    ))
])

pipeline_rf_tratado.fit(X_train_tratado, y_train_tratado)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['EstadoCivil', 'Curso',
                                                   'QualificacaoAnterior',
                                                   'Nacionalidade', 'Genero']),
                                                 ('num', StandardScaler(),
                                                  ['QualificacaoAnteriorGrau',
                                                   'NotaAdmissao',
                                                   'UnidadesCurriculares1SemestreCreditado',
                                                   'UnidadesCurriculares1SemestreInscrito',
                                                   'UnidadesCurricular...
                                                   'UnidadesCurriculares2SemestreGrau',
                      

In [17]:
y_pred_tratado = pipeline_rf_tratado.predict(X_test_tratado)
y_proba_tratado = pipeline_rf_tratado.predict_proba(X_test_tratado)[:, 1]

In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score

metricas_teste = pd.DataFrame([
    {
        "cenario": "Sem tratamento",
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    },
    {
        "cenario": "Com tratamento",
        "accuracy": accuracy_score(y_test_tratado, y_pred_tratado),
        "precision": precision_score(y_test_tratado, y_pred_tratado),
        "recall": recall_score(y_test_tratado, y_pred_tratado),
        "f1": f1_score(y_test_tratado, y_pred_tratado),
        "roc_auc": roc_auc_score(y_test_tratado, y_proba_tratado),
    },
])

metricas_teste

          cenario  accuracy  precision    recall        f1   roc_auc
0  Sem tratamento  0.867797   0.771987  0.834507  0.802030  0.923836
1  Com tratamento  0.870056   0.784512  0.820423  0.802065  0.926466

In [19]:
print("Matriz de confusao - sem tratamento:")
print(confusion_matrix(y_test, y_pred))

print("\nMatriz de confusao - com tratamento:")
print(confusion_matrix(y_test_tratado, y_pred_tratado))

print("\nRelatorio de classificacao - com tratamento:")
print(classification_report(y_test_tratado, y_pred_tratado))

Matriz de confusao - sem tratamento:
[[531  70]
 [ 47 237]]

Matriz de confusao - com tratamento:
[[537  64]
 [ 51 233]]

Relatorio de classificacao - com tratamento:
              precision    recall  f1-score   support

           0       0.91      0.89      0.90       601
           1       0.78      0.82      0.80       284

    accuracy                           0.87       885
   macro avg       0.85      0.86      0.85       885
weighted avg       0.87      0.87      0.87       885



In [20]:
cv_results_tratado = cross_validate(
    pipeline_rf_tratado,
    X_train_tratado,
    y_train_tratado,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

In [21]:
comparacao_cv = []

for metric in scoring.keys():
    comparacao_cv.append({
        "metrica": metric,
        "treino_sem_tratamento": cv_results[f"train_{metric}"].mean(),
        "validacao_sem_tratamento": cv_results[f"test_{metric}"].mean(),
        "treino_com_tratamento": cv_results_tratado[f"train_{metric}"].mean(),
        "validacao_com_tratamento": cv_results_tratado[f"test_{metric}"].mean(),
    })

comparacao_cv = pd.DataFrame(comparacao_cv)
comparacao_cv

     metrica  treino_sem_tratamento  validacao_sem_tratamento  treino_com_tratamento  validacao_com_tratamento
0   accuracy               0.867689                  0.842052               0.873764                  0.848266
1  precision               0.770097                  0.735582               0.786111                  0.749638
2     recall               0.838611                  0.795088               0.834434                  0.793315
3         f1               0.802881                  0.763948               0.809480                  0.770601
4    roc_auc               0.937795                  0.907801               0.938531                  0.907949